#Análise genética embrionária:



*   **PGT-A (Aneuploidia):** Analisa os 24 cromossomas para detetar a falta ou o excesso de cromossomas, uma das principais causas de aborto espontâneo ou de insucesso na fertilização in vitro (FIV).
*   **PGT-M (Monogénico / gene único):** Teste específico para uma mutação hereditária conhecida na família (por exemplo, fibrose quística, distrofia muscular, doença de Huntington).
*   **PGT-SR (Rearranjos estruturais):** Deteta inversões ou translocações cromossómicas que podem provocar problemas genéticos graves no feto.





#Exercício 1

Teste PGT-M

**Grupo 1:** Doenças Autossómicas Recessivas
*   Fibrose Quística, Anemia Falciforme, Doença de Tay-Sachs, Beta-Talassémia, Atrofia Muscular Espinal
* *Genes testados:* CFTR, HBB, HEXA, SMN1

**Grupo 2:** Doenças Autossómicas Dominantes
*  Doença de Huntington, Síndrome de Marfan, Distrofia Miotónica, Neurofibromatose Tipo 1
* *Genes testados:* HTT, FBN1, DMPK, NF1

**Grupo 3:** Doenças ligadas ao cromossoma X
*   Distrofia Muscular de Duchenne, Síndrome do X Frágil, Hemofilia A e B
*   DMD, FMR1, F8, F9

**Grupo 4:** Cancros hereditários
*  BRCA1 e BRCA2 (Mama/Ovário), Síndrome de Lynch
*  BRCA1, BRCA2, MLH1, MSH2

**Grupo 5:** Doenças Metabólicas e de Armazenamento
*  Gaucher, Fenilcetonúria, Galactosemia, Adrenoleucodistrofia
*  GBA, PAH, GALT, ABCD1

**Grupo 6:** Condições Neurológicas e Neuromusculares Específicas
*  Charcot-Marie-Tooth, Osteogénese Imperfeita
*  PMP22, MPZ, TSC1, TSC2

##Exercício 1.1
Retirar a sequência completa dos genes no do NCBI.
Tirar informação clínica do gene e verificar qual a doença relacionada (?)

In [3]:
from Bio import Entrez, SeqIO


# OBRIGATÓRIO: identificares-te ao NCBI
Entrez.email = "o_teu_email@exemplo.com"

# ─── 1. Buscar a sequência completa do gene ───────────────────────────────────
gene_nome = "BRCA1"  # exemplo

# Pesquisar o gene na base de dados "nucleotide"
handle = Entrez.esearch(db="nucleotide", term=f"{gene_nome}[Gene Name] AND Homo sapiens[Organism]", retmax=1)
record = Entrez.read(handle)
handle.close()

gene_id = record["IdList"][0]
print(f"ID encontrado: {gene_id}")

# Descarregar a sequência em formato GenBank
handle = Entrez.efetch(db="nucleotide", id=gene_id, rettype="gb", retmode="text")
seq_record = SeqIO.read(handle, "genbank")
handle.close()

print(f"\nGene: {seq_record.name}")
print(f"Descrição: {seq_record.description}")
print(f"Sequência (primeiros 100 bp): {seq_record.seq[:100]}")

# ─── 2. Informação clínica — pesquisar no OMIM/Gene ──────────────────────────
handle = Entrez.esearch(db="gene", term=f"{gene_nome}[Gene Name] AND Homo sapiens[Organism]", retmax=1)
record = Entrez.read(handle)
handle.close()

gene_db_id = record["IdList"][0]

handle = Entrez.efetch(db="gene", id=gene_db_id, rettype="gene_table", retmode="text")
info = handle.read()
handle.close()

print(f"\nInformação clínica do gene {gene_nome}:")
print(info[:1000])  # primeiros 1000 caracteres

ID encontrado: 262359905

Gene: NG_005905
Descrição: Homo sapiens BRCA1 DNA repair associated (BRCA1), RefSeqGene (LRG_292) on chromosome 17
Sequência (primeiros 100 bp): TGTGTGTATGAAGTTAACTTCAAAGCAAGCTTCCTGTGCTGAGGGGGTGGGAGGTAAGGGTGTGATGAGGCAGGGCTTCTCCTTTGGCAAAGCCTCTGTA

Informação clínica do gene BRCA1:
BRCA1 BRCA1 DNA repair associated[Homo sapiens]
Gene ID: 672, updated on 12-Apr-2026


Reference GRCh38.p14 Primary Assembly NC_000017.11  (minus strand) from: 43170327 to: 43044295
mRNA transcript variant 317 NM_001408458.1, 22 exons,  total annotated spliced exon length: 3785
protein isoform 116 NP_001395387.1, 20 coding  exons,  annotated AA length: 712

Exon table for  mRNA  NM_001408458.1 and protein NP_001395387.1
Genomic Interval Exon		Genomic Interval Coding		Gene Interval Exon		Gene Interval Coding		Exon Length	Coding Length	Intron Length
-------------------------------------------------------------------------------------------------------------------------------------------

##Exercício 1.2 alinhamento de genes

###Exercício 1.2.1
Confirmar se a lista de genes do teste têm alguma modificação. Calcula a similaridade de indentidade e score (?)

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import pairwise2
from Bio.pairwise2 import format_alignment
import pandas as pd

# ──────────────────────────────────────────────
# CONFIGURAÇÃO
# ──────────────────────────────────────────────
Entrez.email = "o_teu_email@exemplo.com"
FICHEIRO_FASTA = "mutacoes_reais_genes.fasta"  # ficheiro com múltiplas sequências
GENE_NOME      = "BRCA1"           # gene a comparar

# ──────────────────────────────────────────────
# 1. LER A SEQUÊNCIA MUTADA DO FICHEIRO FASTA
# ──────────────────────────────────────────────
# Parse o ficheiro FASTA para encontrar o gene desejado
sequencias = list(SeqIO.parse(FICHEIRO_FASTA, "fasta"))
mutada = None

for seq_record in sequencias:
    if GENE_NOME in seq_record.id:
        mutada = seq_record
        break

if mutada is None:
    # Se não encontrar, usar a primeira sequência
    mutada = sequencias[0]
    print(f"⚠️  Gene {GENE_NOME} não encontrado. Usando: {mutada.id}")
else:
    print(f"✅ Sequência carregada: {mutada.id}")

print(f"   Comprimento: {len(mutada.seq)} bp\n")

# ──────────────────────────────────────────────
# 2. BUSCAR A SEQUÊNCIA DE REFERÊNCIA NO NCBI
# ──────────────────────────────────────────────
print("🔍 A buscar sequência de referência no NCBI...")
handle = Entrez.esearch(
    db="nucleotide",
    term=f"{GENE_NOME}[Gene Name] AND Homo sapiens[Organism] AND mRNA[Filter]",
    retmax=1
)
record = Entrez.read(handle)
handle.close()

ref_id = record["IdList"][0]
handle = Entrez.efetch(db="nucleotide", id=ref_id, rettype="fasta", retmode="text")
referencia = SeqIO.read(handle, "fasta")
handle.close()

print(f"✅ Referência obtida: {referencia.id}")
print(f"   Comprimento: {len(referencia.seq)} bp\n")

# ──────────────────────────────────────────────
# 3. ALINHAMENTO LOCAL — % IDENTIDADE + SCORE
# ──────────────────────────────────────────────
print("📐 A calcular alinhamento local (Smith-Waterman)...")

alinhamentos = pairwise2.align.localms(
    str(mutada.seq),
    str(referencia.seq),
    2,    # match score
    -1,   # mismatch penalty
    -2,   # gap open penalty
    -0.5  # gap extend penalty
)

melhor = alinhamentos[0]
seq_alin_mutada = melhor.seqA
seq_alin_ref    = melhor.seqB
score           = melhor.score

# Calcular % identidade manualmente
matches = sum(a == b for a, b in zip(seq_alin_mutada, seq_alin_ref) if a != '-' and b != '-')
total   = sum(1 for a, b in zip(seq_alin_mutada, seq_alin_ref) if a != '-' and b != '-')
identidade = (matches / total) * 100 if total > 0 else 0

print(f"\n📊 RESULTADOS DO ALINHAMENTO:")
print(f"   Alignment Score : {score:.1f}")
print(f"   % Identidade    : {identidade:.2f}%")
print(f"\n{format_alignment(*melhor)}")

# ──────────────────────────────────────────────
# 4. LOCALIZAR AS MUTAÇÕES (posição exata)
# ──────────────────────────────────────────────
print("\n🧬 LOCALIZAÇÃO DAS MUTAÇÕES:")
print(f"{'Posição':<10} {'Ref':<6} {'Mutada':<6} {'Tipo'}")
print("-" * 40)

posicao_real = 0
mutacoes_encontradas = 0

for i, (a, b) in enumerate(zip(seq_alin_mutada, seq_alin_ref)):
    if b != '-':
        posicao_real += 1
    if a != b:
        tipo = "INSERÇÃO" if b == '-' else "DELEÇÃO" if a == '-' else "SNP"
        print(f"{posicao_real:<10} {b:<6} {a:<6} {tipo}")
        mutacoes_encontradas += 1

if mutacoes_encontradas == 0:
    print("   Nenhuma mutação encontrada — sequências idênticas!")
else:
    print(f"\n   Total de diferenças: {mutacoes_encontradas}")

# ──────────────────────────────────────────────
# 5. BLAST ONLINE — E-VALUE
# ──────────────────────────────────────────────
print("\n🌐 A correr BLAST online (pode demorar 1-2 min)...")

result_handle = NCBIWWW.qblast(
    program   = "blastn",
    database  = "nt",
    sequence  = str(mutada.seq),
    hitlist_size = 5
)

blast_records = NCBIXML.read(result_handle)

print(f"\n📊 TOP RESULTADOS BLAST:")
print(f"{'Hit':<40} {'E-value':<15} {'Score':<10} {'Identidade'}")
print("-" * 75)

for alignment in blast_records.alignments[:5]:
    hsp = alignment.hsps[0]  # melhor hsp de cada hit
    ident_pct = (hsp.identities / hsp.align_length) * 100
    print(f"{alignment.title[:38]:<40} {hsp.expect:<15.2e} {hsp.score:<10} {ident_pct:.1f}%")

ValueError: More than one record found in handle

###Exercício 1.2.2
Confirmar resultados por nblast.

##Exercício 1.2 Consequência da mutação


###Exercício 1.3.1
Se sim, converter a sequência para aminoácidos para verificar como a alteração da sequêmcia de nucleotídica altera.

In [ ]:
from Bio.Seq import Seq

# ──────────────────────────────────────────────
# 6. TRADUÇÃO PARA AMINOÁCIDOS
# ──────────────────────────────────────────────
print("\n🔬 TRADUÇÃO PARA AMINOÁCIDOS:")

def traduzir_sequencia(seq, nome):
    """Traduz nucleótidos → aminoácidos, lida com comprimentos não múltiplos de 3"""
    seq_obj = Seq(str(seq))
    # Cortar para múltiplo de 3
    remainder = len(seq_obj) % 3
    if remainder != 0:
        seq_obj = seq_obj[:-remainder]
        print(f"   ⚠️  {nome}: cortados {remainder} bp para alinhar codões")
    return seq_obj.translate(to_stop=True)  # para na stop codon

proteina_ref    = traduzir_sequencia(referencia.seq, "Referência")
proteina_mutada = traduzir_sequencia(mutada.seq,     "Mutada")

print(f"\n   Proteína REFERÊNCIA ({len(proteina_ref)} aa):")
print(f"   {proteina_ref[:80]}{'...' if len(proteina_ref) > 80 else ''}")

print(f"\n   Proteína MUTADA    ({len(proteina_mutada)} aa):")
print(f"   {proteina_mutada[:80]}{'...' if len(proteina_mutada) > 80 else ''}")

# ──────────────────────────────────────────────
# 7. COMPARAR AS PROTEÍNAS — encontrar alterações
# ──────────────────────────────────────────────
print("\n🧬 ALTERAÇÕES NA PROTEÍNA:")
print(f"{'Posição aa':<12} {'Ref':<8} {'Mutada':<8} {'Tipo'}")
print("-" * 45)

# Alinhar as duas proteínas
alin_prot = pairwise2.align.globalms(
    str(proteina_ref),
    str(proteina_mutada),
    2, -1, -2, -0.5
)[0]

prot_alin_ref    = alin_prot.seqA
prot_alin_mutada = alin_prot.seqB

alteracoes = []
pos_real = 0

for i, (ref_aa, mut_aa) in enumerate(zip(prot_alin_ref, prot_alin_mutada)):
    if ref_aa != '-':
        pos_real += 1
    if ref_aa != mut_aa:
        tipo = "INSERÇÃO" if ref_aa == '-' else "DELEÇÃO" if mut_aa == '-' else "MISSENSE"
        alteracoes.append((pos_real, ref_aa, mut_aa, tipo))
        print(f"{pos_real:<12} {ref_aa:<8} {mut_aa:<8} {tipo}")

# Verificar se a proteína ficou mais curta (stop codon prematuro)
diff_comprimento = len(proteina_ref) - len(proteina_mutada)
if diff_comprimento > 0:
    print(f"\n   ⛔ Stop codon prematuro! Proteína mutada é {diff_comprimento} aa mais curta")
    print(f"      → Possível NONSENSE mutation ou frameshift")
elif diff_comprimento < 0:
    print(f"\n   ➕ Proteína mutada é {abs(diff_comprimento)} aa mais longa")

if not alteracoes:
    print("   ✅ Proteína idêntica — mutação é SINÓNIMA (silent mutation)")

# ──────────────────────────────────────────────
# 8. CLASSIFICAR O TIPO DE MUTAÇÃO
# ──────────────────────────────────────────────
print("\n📋 CLASSIFICAÇÃO FINAL DA MUTAÇÃO:")

if not alteracoes and mutacoes_encontradas > 0:
    print("   🟡 SINÓNIMA (Silent) — altera nucleótido mas NÃO altera aminoácido")
elif any(t == "MISSENSE" for _, _, _, t in alteracoes):
    print("   🟠 MISSENSE — troca de aminoácido(s)")
    for pos, ref_aa, mut_aa, _ in alteracoes:
        print(f"      p.{ref_aa}{pos}{mut_aa}")   # notação HGVS proteica
if diff_comprimento > 0:
    print("   🔴 NONSENSE / FRAMESHIFT — proteína truncada")
```

---

## 📊 Exemplo de output esperado
```
🔬 TRADUÇÃO PARA AMINOÁCIDOS:
   Proteína REFERÊNCIA (245 aa):
   MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY...

   Proteína MUTADA    (245 aa):
   MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY...

🧬 ALTERAÇÕES NA PROTEÍNA:
Posição aa   Ref      Mutada   Tipo
---------------------------------------------
12           G        V        MISSENSE

📋 CLASSIFICAÇÃO FINAL DA MUTAÇÃO:
   🟠 MISSENSE — troca de aminoácido(s)
      p.G12V    ← notação HGVS

###Exercício 1.3.2
Confirmar no blastx

##Exercício 1.4 Confirmar patogenecidade da sequência mutada

Usar base de dados X.

#Exercício 2

Anotação de genoma para testar alterações cromossomais:

*   **PGT-A (Aneuploidia):** Analisa os 24 cromossomas para detetar a falta ou o excesso de cromossomas, uma das principais causas de aborto espontâneo ou de insucesso na fertilização in vitro (FIV).
** PGT-SR (Rearranjos estruturais):** Deteta inversões ou translocações cromossómicas que podem provocar problemas genéticos graves no feto.

**Grupo 1:** Doenças Autossómicas Recessivas
*   Cromossoma 21, Trissomia 21 (Down Syndrome)

**Grupo 2:** Doenças Autossómicas Dominantes
*   Cromossoma 18, Trissomia 18 (Edwards Syndrome)

**Grupo 3:** Doenças ligadas ao cromossoma X
*   Cromossoma 13, Trissomia 13 (Patau Syndrome)

**Grupo 4:** Cancros hereditários
*  Cromossomas XXY, Klinefelter Syndrome (XXY)

**Grupo 5:** Doenças Metabólicas e de Armazenamento
*  Cromossomas XXY, Turner Syndrome (XO)

**Grupo 6:** Condições Neurológicas e Neuromusculares Específicas
*  Cromossomas XYY, Syndrome (Jacob's Syndrome):

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ──────────────────────────────────────────────
# CONFIGURAÇÃO
# ──────────────────────────────────────────────
Entrez.email = "o_teu_email@exemplo.com"

# Usamos hg38 (GRCh38) — mais recente e recomendado
GENOME_REF = "GRCh38"

# Grupos do exercício
CROMOSSOMAS = {
    "Grupo 1 - Autossómica Recessiva": {
        "chr": "21", "sindrome": "Down Syndrome",       "tipo": "Trissomia", "esperado": 3
    },
    "Grupo 2 - Autossómica Dominante": {
        "chr": "18", "sindrome": "Edwards Syndrome",    "tipo": "Trissomia", "esperado": 3
    },
    "Grupo 3 - Ligada ao X": {
        "chr": "13", "sindrome": "Patau Syndrome",      "tipo": "Trissomia", "esperado": 3
    },
    "Grupo 4 - Cancros Hereditários": {
        "chr": "X",  "sindrome": "Klinefelter Syndrome","tipo": "XXY",       "esperado": 2
    },
    "Grupo 5 - Metabólica": {
        "chr": "X",  "sindrome": "Turner Syndrome",     "tipo": "XO",        "esperado": 1
    },
    "Grupo 6 - Neurológica": {
        "chr": "Y",  "sindrome": "Jacob's Syndrome",    "tipo": "XYY",       "esperado": 2
    },
}

# ──────────────────────────────────────────────
# 1. BUSCAR SEQUÊNCIAS DE REFERÊNCIA NO NCBI
# ──────────────────────────────────────────────
print("🔍 A buscar cromossomas no NCBI (GRCh38)...\n")

def buscar_cromossoma_ncbi(numero_chr):
    """Busca a sequência de referência de um cromossoma no NCBI."""
    termo = f"Homo sapiens chromosome {numero_chr} {GENOME_REF} reference assembly"
    handle = Entrez.esearch(db="nucleotide", term=termo, retmax=1)
    record = Entrez.read(handle)
    handle.close()

    if not record["IdList"]:
        print(f"   ⚠️  Cromossoma {numero_chr} não encontrado!")
        return None

    ncbi_id = record["IdList"][0]
    handle = Entrez.efetch(
        db="nucleotide", id=ncbi_id,
        rettype="fasta", retmode="text",
        seq_start=1, seq_stop=10000  # primeiros 10kb para não sobrecarregar
    )
    seq = SeqIO.read(handle, "fasta")
    handle.close()
    return seq, ncbi_id

# ──────────────────────────────────────────────
# 2. PGT-A — CONTAR CÓPIAS DE CROMOSSOMAS
# ──────────────────────────────────────────────
print("=" * 60)
print("📊 PGT-A — ANÁLISE DE ANEUPLOIDIA")
print("=" * 60)

resultados_pgta = []

for grupo, info in CROMOSSOMAS.items():
    chr_num = info["chr"]
    print(f"\n🔬 {grupo} — Cromossoma {chr_num} ({info['sindrome']})")

    resultado = buscar_cromossoma_ncbi(chr_num)
    if resultado is None:
        continue

    seq, ncbi_id = resultado
    comprimento = len(seq.seq)

    # Simular número de cópias baseado no tipo de síndrome
    # (em ambiente real, isto viria de dados NGS/array CGH)
    copias_simuladas = info["esperado"]
    normal = 2  # diplóide normal

    if copias_simuladas > normal:
        status = f"⚠️  TRISSOMIA (n={copias_simuladas})"
        cor = "red"
    elif copias_simuladas < normal:
        status = f"⚠️  MONOSSOMIA (n={copias_simuladas})"
        cor = "orange"
    else:
        status = f"✅ NORMAL (n={copias_simuladas})"
        cor = "green"

    print(f"   NCBI ID      : {ncbi_id}")
    print(f"   Comprimento  : {comprimento:,} bp (amostra 10kb)")
    print(f"   Cópias       : {status}")
    print(f"   Síndrome     : {info['sindrome']} ({info['tipo']})")

    resultados_pgta.append({
        "Grupo"       : grupo,
        "Cromossoma"  : f"Chr {chr_num}",
        "Síndrome"    : info["sindrome"],
        "Tipo"        : info["tipo"],
        "Cópias"      : copias_simuladas,
        "Status"      : status.split("(")[0].strip(),
        "NCBI ID"     : ncbi_id
    })

# ──────────────────────────────────────────────
# 3. PGT-SR — DETETAR REARRANJOS ESTRUTURAIS
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("🔬 PGT-SR — ANÁLISE DE REARRANJOS ESTRUTURAIS")
print("=" * 60)

def analisar_rearranjos(seq, chr_num):
    """
    Deteta potenciais inversões/translocações comparando
    padrões na sequência de referência.
    """
    seq_str = str(seq.seq).upper()
    tamanho  = len(seq_str)

    # Dividir em janelas para análise de rearranjos
    janela = tamanho // 4
    segmentos = [seq_str[i:i+janela] for i in range(0, tamanho, janela)]

    print(f"\n   Cromossoma {chr_num} — {tamanho:,} bp analisados")
    print(f"   {'Segmento':<12} {'Tamanho':<12} {'GC%':<10} {'Status'}")
    print("   " + "-" * 50)

    anomalias = []
    gc_conteudos = []

    for i, seg in enumerate(segmentos[:4]):
        gc = (seg.count("G") + seg.count("C")) / len(seg) * 100 if seg else 0
        gc_conteudos.append(gc)

        # Sinalizar se GC% anómalo (pode indicar rearranjo)
        if gc < 30 or gc > 60:
            stat = "⚠️  GC% anómalo"
            anomalias.append(i)
        else:
            stat = "✅ Normal"

        print(f"   Segmento {i+1:<4} {len(seg):<12,} {gc:<10.1f} {stat}")

    # Verificar variação entre segmentos (possível indicador de rearranjo)
    variacao_gc = np.std(gc_conteudos)
    if variacao_gc > 10:
        print(f"\n   ⚠️  Alta variação de GC% entre segmentos (σ={variacao_gc:.1f})")
        print(f"      → Possível INVERSÃO ou TRANSLOCAÇÃO")
    else:
        print(f"\n   ✅ GC% estável (σ={variacao_gc:.1f}) — sem rearranjos evidentes")

    return anomalias

for grupo, info in CROMOSSOMAS.items():
    chr_num = info["chr"]
    resultado = buscar_cromossoma_ncbi(chr_num)
    if resultado:
        seq, _ = resultado
        analisar_rearranjos(seq, chr_num)

# ──────────────────────────────────────────────
# 4. TABELA RESUMO
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("📋 TABELA RESUMO — PGT-A")
print("=" * 60)
df = pd.DataFrame(resultados_pgta)
print(df[["Cromossoma", "Síndrome", "Tipo", "Cópias", "Status"]].to_string(index=False))

# ──────────────────────────────────────────────
# 5. GRÁFICO — NÚMERO DE CÓPIAS POR CROMOSSOMA
# ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

cores = {
    "⚠️  TRISSOMIA" : "#e74c3c",
    "⚠️  MONOSSOMIA": "#e67e22",
    "✅ NORMAL"     : "#2ecc71"
}

labels = [f"Chr {r['Cromossoma'].split()[1]}\n{r['Síndrome']}" for r in resultados_pgta]
copias = [r["Cópias"] for r in resultados_pgta]
status = [r["Status"] for r in resultados_pgta]
bar_cores = [cores.get(s, "#3498db") for s in status]

bars = ax.bar(labels, copias, color=bar_cores, edgecolor="white", linewidth=1.5, width=0.6)
ax.axhline(y=2, color="navy", linestyle="--", linewidth=1.5, label="Normal (n=2)")

for bar, val in zip(bars, copias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha="center", va="bottom", fontweight="bold", fontsize=11)

# Legenda
patches = [mpatches.Patch(color=c, label=l) for l, c in cores.items()]
ax.legend(handles=patches, loc="upper right")

ax.set_title("PGT-A — Número de Cópias por Cromossoma", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Número de Cópias")
ax.set_ylim(0, 4.5)
ax.set_yticks([0, 1, 2, 3, 4])
plt.tight_layout()
plt.savefig("pgta_resultado.png", dpi=150)
plt.show()
print("\n✅ Gráfico guardado como 'pgta_resultado.png'")
```

---

## 📊 Output esperado
```
📊 PGT-A — ANÁLISE DE ANEUPLOIDIA
════════════════════════════════════════
🔬 Grupo 1 — Cromossoma 21 (Down Syndrome)
   Cópias : ⚠️  TRISSOMIA (n=3)

🔬 Grupo 5 — Cromossoma X (Turner Syndrome)
   Cópias : ⚠️  MONOSSOMIA (n=1)

🔬 PGT-SR — REARRANJOS ESTRUTURAIS
   Segmento 1    2500    41.2    ✅ Normal
   Segmento 2    2500    58.9    ⚠️  GC% anómalo
   → Possível INVERSÃO ou TRANSLOCAÇÃO